# reward_v2 — исправленная reward-функция + честный Optimal-оракул



## 1. Setup

In [2]:
import numpy as np
from scipy import stats
import json

EMOTIONS = ['anxiety', 'burnout', 'sadness', 'overwhelm', 'neutral']
EMOTION_GAMES = {
    'anxiety':   [0, 1, 2],
    'burnout':   [3, 4, 5],
    'neutral':   [6, 7, 8],
    'overwhelm': [9, 10, 11],
    'sadness':   [12, 13, 14],
}
EMOTION_TO_IDX = {'anxiety': 0, 'burnout': 1, 'sadness': 2, 'overwhelm': 3, 'neutral': 4}

REWARD_TABLE = {
    'anxiety':   [0.90,0.75,0.70,0.30,0.20,0.25,0.20,0.15,0.10,0.15,0.20,0.25,0.20,0.30,0.25],
    'burnout':   [0.35,0.25,0.30,0.90,0.75,0.80,0.30,0.25,0.20,0.20,0.25,0.30,0.40,0.55,0.45],
    'neutral':   [0.25,0.30,0.20,0.40,0.45,0.40,0.85,0.80,0.75,0.35,0.40,0.30,0.35,0.40,0.35],
    'overwhelm': [0.50,0.45,0.40,0.35,0.25,0.30,0.30,0.25,0.20,0.85,0.80,0.75,0.25,0.30,0.20],
    'sadness':   [0.25,0.20,0.25,0.45,0.35,0.40,0.30,0.25,0.20,0.25,0.30,0.20,0.60,0.75,0.85],
}

N_ARMS, STATE_DIM = 15, 6
STRESS_RANGES = {
    'anxiety':(0.7,1.0), 'burnout':(0.6,0.9), 'sadness':(0.5,0.9),
    'overwhelm':(0.7,1.0), 'neutral':(0.0,0.4)
}

def make_user_offsets(n_users, offset_std, seed):
    rng = np.random.RandomState(seed)
    return {uid: rng.normal(0, offset_std, N_ARMS) for uid in range(n_users)}

def make_vector(emotion, stress):
    oh = [0.0]*5
    oh[EMOTION_TO_IDX[emotion]] = 1.0
    return np.array(oh + [stress])

print("Setup OK")

Setup OK


## 2. reward_v2 — Decision A

1. `click` генерируется отдельно (через `click_u` vs `engagement_prob`)
2. `click == 0` → `playtime = 0` жёстко, без случайности
3. `click == 1` → `playtime` сэмплируется через `sample_duration_given_click`
4. `bounce = 1`, если `click==1 and playtime < 0.5` (30 сек)
5. Reward считается после того, как известны `click`/`playtime`/`bounce`

Дизайн специально без "серой зоны": каждая ветка `sample_duration_given_click`
однозначно решает bounce/no-bounce (диапазоны `[0, 0.5)` и `[0.5, 15]` не пересекаются).

In [3]:
def sample_duration_given_click(engagement_prob, rng):
    if rng.random() < engagement_prob:
        return rng.uniform(0.5, 15.0)   # sustained: всегда >= 30 сек, никогда не bounce
    else:
        return rng.uniform(0.0, 0.5)    # post-click bounce: всегда < 30 сек, всегда bounce


def reward_v2(base_reward, noise_z, click_u, rng, noise_std=0.08):
    """
    base_reward : REWARD_TABLE[emotion][arm] (+ user_offset[arm], если применимо)
    noise_z     : N(0,1) шум, добавляемый к engagement_prob
    click_u     : Uniform(0,1), решает click да/нет
    rng         : отдельный генератор для sample_duration (не должен мешать master_rng)
    """
    engagement_prob = np.clip(base_reward + noise_z * noise_std, 0, 1)
    click = 1 if click_u < engagement_prob else 0

    if click == 0:
        playtime = 0.0
        bounce = 1
    else:
        playtime = sample_duration_given_click(engagement_prob, rng)
        bounce = 1 if playtime < 0.5 else 0

    if click == 0 or bounce:
        R = -0.7
    else:
        R = round(0.3 + 0.7 * min(1.0, playtime / 10), 3)

    return R, click, playtime, bounce, engagement_prob


print("reward_v2 defined")

reward_v2 defined


### 2.1 Тесты reward_v2

In [14]:
rng = np.random.RandomState(0)

print("Тест 1: click=0 -> playtime=0 -> R=-0.7 ДЕТЕРМИНИРОВАННО")
for click_u_test in [0.01, 0.3, 0.5, 0.7, 0.99]:
    R, click, playtime, bounce, eng = reward_v2(
        base_reward=0.0, noise_z=0.0, click_u=click_u_test, rng=rng
    )
    assert click == 0
    assert playtime == 0.0
    assert bounce == 1
    assert R == -0.7
    print(f"  click_u={click_u_test}: click={click} playtime={playtime} bounce={bounce} R={R}  [OK]")

print("\nТест 2: диапазоны bounce/sustain playtime не пересекаются")
test_rng = np.random.RandomState(42)
playtimes_bounce, playtimes_sustain = [], []
for _ in range(10000):
    R, click, playtime, bounce, eng = reward_v2(
        base_reward=0.5, noise_z=0.0, click_u=0.3, rng=test_rng
    )
    if click == 1:
        (playtimes_bounce if bounce else playtimes_sustain).append(playtime)
print(f"  bounce range: [{min(playtimes_bounce):.3f}, {max(playtimes_bounce):.3f}]")
print(f"  sustain range: [{min(playtimes_sustain):.3f}, {max(playtimes_sustain):.3f}]")
assert max(playtimes_bounce) < 0.5
assert min(playtimes_sustain) >= 0.5
print("  [OK] Пересечения нет.")

print("\nТест 3: диапазон R -- ровно {-0.7} U [0.3, 1.0], без промежуточных значений")
test_rng = np.random.RandomState(7)
all_R = []
for _ in range(5000):
    base = test_rng.uniform(0, 1)
    noise_z = test_rng.normal()
    click_u = test_rng.uniform()
    R, *_ = reward_v2(base, noise_z, click_u, test_rng)
    all_R.append(R)
all_R = np.array(all_R)
print(f"  R == -0.7: {np.sum(all_R == -0.7)} раз из 5000")
print(f"  R > 0: min={all_R[all_R>0].min():.3f} max={all_R[all_R>0].max():.3f}")
assert np.sum((all_R > -0.7) & (all_R < 0.3)) == 0
print("  [OK] Нет значений в 'ничейной' зоне между -0.7 и 0.3.")

print("\nВСЕ ТЕСТЫ reward_v2 ПРОШЛИ")

Тест 1: click=0 -> playtime=0 -> R=-0.7 ДЕТЕРМИНИРОВАННО
  click_u=0.01: click=0 playtime=0.0 bounce=1 R=-0.7  [OK]
  click_u=0.3: click=0 playtime=0.0 bounce=1 R=-0.7  [OK]
  click_u=0.5: click=0 playtime=0.0 bounce=1 R=-0.7  [OK]
  click_u=0.7: click=0 playtime=0.0 bounce=1 R=-0.7  [OK]
  click_u=0.99: click=0 playtime=0.0 bounce=1 R=-0.7  [OK]

Тест 2: диапазоны bounce/sustain playtime не пересекаются
  bounce range: [0.000, 0.500]
  sustain range: [0.500, 14.995]
  [OK] Пересечения нет.

Тест 3: диапазон R -- ровно {-0.7} U [0.3, 1.0], без промежуточных значений
  R == -0.7: 3330 раз из 5000
  R > 0: min=0.335 max=1.000
  [OK] Нет значений в 'ничейной' зоне между -0.7 и 0.3.

ВСЕ ТЕСТЫ reward_v2 ПРОШЛИ


## 3. Optimal vs Population-best



In [4]:
def population_best_action(emotion, valid_arms):
    vals = [REWARD_TABLE[emotion][a] for a in valid_arms]
    return valid_arms[int(np.argmax(vals))]

def optimal_action(emotion, valid_arms, user_offset):
    vals = [REWARD_TABLE[emotion][a] + (user_offset[a] if user_offset is not None else 0.0)
            for a in valid_arms]
    return valid_arms[int(np.argmax(vals))]

print("Тест: offset_std=0 -> Optimal == Population-best для каждой эмоции")
zero_offset = np.zeros(15)
for emotion in EMOTIONS:
    valid = EMOTION_GAMES[emotion]
    pb = population_best_action(emotion, valid)
    opt = optimal_action(emotion, valid, zero_offset)
    assert pb == opt, f"MISMATCH at {emotion}: pb={pb} opt={opt}"
    print(f"  {emotion}: population_best={pb} optimal={opt} [OK]")

print("\nТест: offset_std>0 -> Optimal персонализирован, Population-best -- нет")
rng2 = np.random.RandomState(42)
emotion = 'anxiety'
valid = EMOTION_GAMES[emotion]
pb = population_best_action(emotion, valid)
choices = set()
for uid in range(10):
    offset = rng2.normal(0, 0.5, 15)
    choices.add(optimal_action(emotion, valid, offset))
print(f"  Population-best: arm {pb} (одинаков для всех)")
print(f"  Optimal-выборы по 10 юзерам: {sorted(choices)}")
print("\nВСЕ ТЕСТЫ Optimal/Population-best ПРОШЛИ")

Тест: offset_std=0 -> Optimal == Population-best для каждой эмоции
  anxiety: population_best=0 optimal=0 [OK]
  burnout: population_best=3 optimal=3 [OK]
  sadness: population_best=14 optimal=14 [OK]
  overwhelm: population_best=9 optimal=9 [OK]
  neutral: population_best=6 optimal=6 [OK]

Тест: offset_std>0 -> Optimal персонализирован, Population-best -- нет
  Population-best: arm 0 (одинаков для всех)
  Optimal-выборы по 10 юзерам: [0, 1, 2]

ВСЕ ТЕСТЫ Optimal/Population-best ПРОШЛИ


## 4. LinUCB bandit

In [5]:
class LinUCBBandit:
    def __init__(self, alpha=0.5, n_arms=N_ARMS, state_dim=STATE_DIM):
        self.alpha = alpha
        self.A = [np.eye(state_dim) for _ in range(n_arms)]
        self.b = [np.zeros(state_dim) for _ in range(n_arms)]

    def select(self, x, valid):
        scores = {}
        for arm in valid:
            A_inv = np.linalg.inv(self.A[arm])
            theta = A_inv @ self.b[arm]
            scores[arm] = theta @ x + self.alpha * np.sqrt(x @ A_inv @ x)
        return max(scores, key=scores.get)

    def update(self, arm, x, r):
        self.A[arm] += np.outer(x, x)
        self.b[arm] += r * x


def clone_bandit(b):
    nb = LinUCBBandit(alpha=b.alpha)
    nb.A = [a.copy() for a in b.A]
    nb.b = [bb.copy() for bb in b.b]
    return nb


def pretrain_population_bandit_v2(n_episodes, seed, alpha=0.5):
    rng = np.random.RandomState(seed)
    bandit = LinUCBBandit(alpha=alpha)
    for _ in range(n_episodes):
        emotion = rng.choice(EMOTIONS)
        low, high = STRESS_RANGES[emotion]
        stress = rng.uniform(low, high)
        x = make_vector(emotion, stress)
        valid = EMOTION_GAMES[emotion]
        a = rng.choice(valid)
        base = REWARD_TABLE[emotion][a]
        z, u = rng.normal(), rng.uniform()
        r, *_ = reward_v2(base, z, u, rng)
        bandit.update(a, x, r)
    return bandit

print("LinUCB ready")

LinUCB ready


## 5. Полный прогон (5 policies: random, pop_best, non_pers, pers, optimal)

CRN применяется ко всем пяти policies одновременно: один и тот же набор шумов
(`noise[arm]`) на раунд, независимо от того, кто какой arm выбрал.

In [7]:
def run_seed_full(seed, n_users, n_sessions, offset_std, warm_start_pers, alpha=0.5, pretrain_episodes=5000):
    master_rng = np.random.RandomState(seed)
    user_offsets = make_user_offsets(n_users, offset_std, seed=seed)
    pretrained = pretrain_population_bandit_v2(pretrain_episodes, seed=seed, alpha=alpha)
    non_pers = clone_bandit(pretrained)
    pers = {uid: (clone_bandit(pretrained) if warm_start_pers else LinUCBBandit(alpha=alpha))
            for uid in range(n_users)}

    rewards = {'random': [], 'pop_best': [], 'non_pers': [], 'pers': [], 'optimal': []}
    for uid in range(n_users):
        offset = user_offsets[uid]
        for s in range(n_sessions):
            emotion = master_rng.choice(EMOTIONS)
            low, high = STRESS_RANGES[emotion]
            stress = master_rng.uniform(low, high)
            x = make_vector(emotion, stress)
            valid = EMOTION_GAMES[emotion]

            # CRN: один набор шумов на раунд для ВСЕХ policies, включая Optimal
            noise = {arm: dict(z=master_rng.normal(), u=master_rng.uniform()) for arm in valid}

            def rf(arm):
                n = noise[arm]
                base = REWARD_TABLE[emotion][arm] + offset[arm]
                local_rng = np.random.RandomState(hash((seed, uid, s, arm)) % (2**31))
                r, *_ = reward_v2(base, n['z'], n['u'], local_rng)
                return r

            a = master_rng.choice(valid); rewards['random'].append(rf(a))
            a = population_best_action(emotion, valid); rewards['pop_best'].append(rf(a))
            a = non_pers.select(x, valid); r = rf(a); non_pers.update(a, x, r); rewards['non_pers'].append(r)
            a = pers[uid].select(x, valid); r = rf(a); pers[uid].update(a, x, r); rewards['pers'].append(r)
            a = optimal_action(emotion, valid, offset); rewards['optimal'].append(rf(a))

    return {k: float(np.mean(v)) for k, v in rewards.items()}

print("run_seed_full ready")

run_seed_full ready


## 6. Run 1 — Original (оба бандита warm-started)

Должен воспроизвести null-результат для personalisation, но теперь под честным reward_v2.

In [8]:
SEEDS_5 = [42, 123, 456, 789, 2024]
SEEDS_20 = SEEDS_5 + [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15]
N_USERS, N_SESSIONS, OFFSET_STD = 30, 100, 0.30

print("="*70)
print("ORIGINAL (both warm-started), reward_v2, n=5 seeds")
print("="*70)

original_results = {k: [] for k in ['random','pop_best','non_pers','pers','optimal']}
for sd in SEEDS_5:
    r = run_seed_full(sd, N_USERS, N_SESSIONS, OFFSET_STD, warm_start_pers=True)
    for k in original_results:
        original_results[k].append(r[k])

for k, v in original_results.items():
    print(f"  {k:10s} mean={np.mean(v):.4f} std={np.std(v):.4f}")

t_orig, p_orig = stats.ttest_rel(original_results['pers'], original_results['non_pers'])
print(f"\npers vs non_pers (paired): t={t_orig:.3f} p={p_orig:.4f}")

ORIGINAL (both warm-started), reward_v2, n=5 seeds
  random     mean=0.1868 std=0.0131
  pop_best   mean=0.3224 std=0.0376
  non_pers   mean=0.3293 std=0.0437
  pers       mean=0.3246 std=0.0389
  optimal    mean=0.6020 std=0.0067

pers vs non_pers (paired): t=-1.002 p=0.3730


## 7. Run 2 — Corrected (personalised cold-started + CRN)

5 confirmatory seeds + 20 extended seeds, paired t-test, permutation test, Holm-Bonferroni,
и проверка: Optimal никогда не должен проигрывать Personalised LinUCB.

In [9]:
print("="*70)
print("CORRECTED (personalised cold-started), reward_v2, n=5 + n=20 seeds")
print("="*70)

corrected_5 = {k: [] for k in ['random','pop_best','non_pers','pers','optimal']}
corrected_20 = {k: [] for k in ['random','pop_best','non_pers','pers','optimal']}

for i, sd in enumerate(SEEDS_20):
    r = run_seed_full(sd, N_USERS, N_SESSIONS, OFFSET_STD, warm_start_pers=False)
    for k in corrected_20:
        corrected_20[k].append(r[k])
        if i < 5:
            corrected_5[k].append(r[k])

print("--- n=5 (confirmatory) ---")
for k, v in corrected_5.items():
    print(f"  {k:10s} mean={np.mean(v):.4f} std={np.std(v):.4f}")
t5, p5 = stats.ttest_rel(corrected_5['pers'], corrected_5['non_pers'])
print(f"  pers vs non_pers (paired, n=5): t={t5:.3f} p={p5:.4f}")

print("\n--- n=20 (extended) ---")
for k, v in corrected_20.items():
    print(f"  {k:10s} mean={np.mean(v):.4f} std={np.std(v):.4f}")
t20, p20 = stats.ttest_rel(corrected_20['pers'], corrected_20['non_pers'])
diffs20 = np.array(corrected_20['pers']) - np.array(corrected_20['non_pers'])
d20 = diffs20.mean() / diffs20.std(ddof=1)
ci = stats.t.interval(0.95, len(diffs20)-1, loc=diffs20.mean(), scale=stats.sem(diffs20))
print(f"  pers vs non_pers (paired, n=20): t={t20:.3f} p={p20:.8f} Cohen_d={d20:.3f}")
print(f"  95% CI on mean paired difference: [{ci[0]:.4f}, {ci[1]:.4f}]")

CORRECTED (personalised cold-started), reward_v2, n=5 + n=20 seeds
--- n=5 (confirmatory) ---
  random     mean=0.1868 std=0.0131
  pop_best   mean=0.3224 std=0.0376
  non_pers   mean=0.3293 std=0.0437
  pers       mean=0.5118 std=0.0090
  optimal    mean=0.6020 std=0.0067
  pers vs non_pers (paired, n=5): t=9.000 p=0.0008

--- n=20 (extended) ---
  random     mean=0.1969 std=0.0292
  pop_best   mean=0.3269 std=0.0327
  non_pers   mean=0.3289 std=0.0338
  pers       mean=0.5076 std=0.0211
  optimal    mean=0.5988 std=0.0180
  pers vs non_pers (paired, n=20): t=24.994 p=0.00000000 Cohen_d=5.589
  95% CI on mean paired difference: [0.1637, 0.1936]


In [10]:
# Permutation test (sign-flip, non-parametric robustness check)
def permutation_test(a, b, n_perm=10000, seed=0):
    rng = np.random.RandomState(seed)
    diffs = np.array(a) - np.array(b)
    obs = diffs.mean()
    count = 0
    for _ in range(n_perm):
        signs = rng.choice([-1, 1], size=len(diffs))
        if abs(np.mean(diffs * signs)) >= abs(obs):
            count += 1
    return obs, count / n_perm

obs_diff, p_perm = permutation_test(corrected_20['pers'], corrected_20['non_pers'])
print(f"Permutation test (n=20, 10000 permutations): obs_diff={obs_diff:.4f} p={p_perm:.4f}")

Permutation test (n=20, 10000 permutations): obs_diff=0.1787 p=0.0000


In [11]:
# Holm-Bonferroni correction across 4 pairwise comparisons
comparisons = {
    'pers_vs_nonpers':    (corrected_20['pers'],     corrected_20['non_pers']),
    'nonpers_vs_random':  (corrected_20['non_pers'], corrected_20['random']),
    'popbest_vs_random':  (corrected_20['pop_best'], corrected_20['random']),
    'nonpers_vs_popbest': (corrected_20['non_pers'], corrected_20['pop_best']),
}
pvals = {}
for name, (a, b) in comparisons.items():
    t, p = stats.ttest_rel(a, b)
    pvals[name] = p

sorted_p = sorted(pvals.items(), key=lambda x: x[1])
print("Holm-Bonferroni correction (n=20):")
m = len(sorted_p)
for i, (name, p) in enumerate(sorted_p):
    holm_alpha = 0.05 / (m - i)
    sig = "significant" if p < holm_alpha else "NOT significant"
    print(f"  {name:22s} p={p:.8f}  Holm threshold={holm_alpha:.4f}  -> {sig}")

Holm-Bonferroni correction (n=20):
  pers_vs_nonpers        p=0.00000000  Holm threshold=0.0125  -> significant
  nonpers_vs_random      p=0.00000000  Holm threshold=0.0167  -> significant
  popbest_vs_random      p=0.00000000  Holm threshold=0.0250  -> significant
  nonpers_vs_popbest     p=0.40966908  Holm threshold=0.0500  -> NOT significant


In [12]:
# Проверка: Optimal никогда не должен проигрывать Personalised LinUCB
violations = sum(1 for o, pe in zip(corrected_20['optimal'], corrected_20['pers']) if o < pe)
print(f"Optimal < Personalised LinUCB: {violations}/{len(SEEDS_20)} seeds")
if violations == 0:
    print("[OK] Optimal всегда доминирует -- честный оракул работает корректно.")
else:
    print("[WARNING] Есть нарушения -- нужно разбираться.")

Optimal < Personalised LinUCB: 0/20 seeds
[OK] Optimal всегда доминирует -- честный оракул работает корректно.


## 8. Итоговая сводка для манускрипта

In [13]:
summary = {
    'reward_version': 'reward_v2 (Decision A: click generated separately, click=0 -> playtime=0 deterministically)',
    'original_warmstart_5seeds': {k: {'mean': round(np.mean(v),4), 'std': round(np.std(v),4)}
                                    for k, v in original_results.items()},
    'original_pers_vs_nonpers_paired': {'t': round(t_orig,3), 'p': round(p_orig,4)},
    'corrected_coldstart_5seeds': {k: {'mean': round(np.mean(v),4), 'std': round(np.std(v),4)}
                                     for k, v in corrected_5.items()},
    'corrected_coldstart_20seeds': {k: {'mean': round(np.mean(v),4), 'std': round(np.std(v),4)}
                                      for k, v in corrected_20.items()},
    'corrected_pers_vs_nonpers_paired_n5': {'t': round(t5,3), 'p': round(p5,4)},
    'corrected_pers_vs_nonpers_paired_n20': {'t': round(t20,3), 'p': p20, 'cohen_d': round(d20,3),
                                               'ci95': [round(ci[0],4), round(ci[1],4)]},
    'permutation_test_n20': {'obs_diff': round(obs_diff,4), 'p': p_perm},
    'holm_bonferroni_n20': {name: round(p,8) for name, p in pvals.items()},
    'optimal_violations_n20': violations,
}

print(json.dumps(summary, indent=2))

with open('reward_v2_results.json', 'w') as f:
    json.dump(summary, f, indent=2)
print("\nSaved reward_v2_results.json")

{
  "reward_version": "reward_v2 (Decision A: click generated separately, click=0 -> playtime=0 deterministically)",
  "original_warmstart_5seeds": {
    "random": {
      "mean": 0.1868,
      "std": 0.0131
    },
    "pop_best": {
      "mean": 0.3224,
      "std": 0.0376
    },
    "non_pers": {
      "mean": 0.3293,
      "std": 0.0437
    },
    "pers": {
      "mean": 0.3246,
      "std": 0.0389
    },
    "optimal": {
      "mean": 0.602,
      "std": 0.0067
    }
  },
  "original_pers_vs_nonpers_paired": {
    "t": -1.002,
    "p": 0.373
  },
  "corrected_coldstart_5seeds": {
    "random": {
      "mean": 0.1868,
      "std": 0.0131
    },
    "pop_best": {
      "mean": 0.3224,
      "std": 0.0376
    },
    "non_pers": {
      "mean": 0.3293,
      "std": 0.0437
    },
    "pers": {
      "mean": 0.5118,
      "std": 0.009
    },
    "optimal": {
      "mean": 0.602,
      "std": 0.0067
    }
  },
  "corrected_coldstart_20seeds": {
    "random": {
      "mean": 0.1969,
      

In [15]:
def pretrain_population_bandit_sensitivity(n_episodes, seed, alpha, noise_std):
    rng = np.random.RandomState(seed)
    bandit = LinUCBBandit(alpha=alpha)
    for _ in range(n_episodes):
        emotion = rng.choice(EMOTIONS)
        low, high = STRESS_RANGES[emotion]
        stress = rng.uniform(low, high)
        x = make_vector(emotion, stress)
        valid = EMOTION_GAMES[emotion]
        a = rng.choice(valid)
        base = REWARD_TABLE[emotion][a]
        z, u = rng.normal(), rng.uniform()
        r, *_ = reward_v2(base, z, u, rng, noise_std=noise_std)
        bandit.update(a, x, r)
    return bandit


def run_paired_delta_v2(n_users, n_sessions, offset_std, alpha, noise_std, seeds):
    """Corrected-style run (personalised cold-started, CRN), возвращает per-seed delta (pers-non_pers)."""
    deltas = []
    for seed in seeds:
        master_rng = np.random.RandomState(seed)
        user_offsets = make_user_offsets(n_users, offset_std, seed=seed)
        pretrained = pretrain_population_bandit_sensitivity(5000, seed=seed, alpha=alpha, noise_std=noise_std)
        non_pers = clone_bandit(pretrained)
        pers = {uid: LinUCBBandit(alpha=alpha) for uid in range(n_users)}  # cold-start

        r_non, r_pers = [], []
        for uid in range(n_users):
            offset = user_offsets[uid]
            for s in range(n_sessions):
                emotion = master_rng.choice(EMOTIONS)
                low, high = STRESS_RANGES[emotion]
                stress = master_rng.uniform(low, high)
                x = make_vector(emotion, stress)
                valid = EMOTION_GAMES[emotion]
                noise = {arm: dict(z=master_rng.normal(), u=master_rng.uniform()) for arm in valid}

                def rf(arm):
                    n = noise[arm]
                    base = REWARD_TABLE[emotion][arm] + offset[arm]
                    local_rng = np.random.RandomState(hash((seed, uid, s, arm)) % (2**31))
                    r, *_ = reward_v2(base, n['z'], n['u'], local_rng, noise_std=noise_std)
                    return r

                a = non_pers.select(x, valid); r = rf(a); non_pers.update(a, x, r); r_non.append(r)
                a = pers[uid].select(x, valid); r = rf(a); pers[uid].update(a, x, r); r_pers.append(r)

        deltas.append(np.mean(r_pers) - np.mean(r_non))

    deltas = np.array(deltas)
    t, p = stats.ttest_1samp(deltas, 0)
    return deltas.mean(), deltas.std(), p

print("Sensitivity functions ready")

Sensitivity functions ready


In [16]:
sweep_results = []

print("=== Sensitivity sweep под reward_v2 ===")
print("baseline: alpha=0.5, noise=0.08, offset_std=0.30, users=30, sessions=100\n")

print("--- alpha ---")
for alpha in [0.1, 0.3, 0.5, 0.8, 1.2]:
    m, s, p = run_paired_delta_v2(30, 100, 0.30, alpha, 0.08, SEEDS_5)
    sig = "*" if p < 0.05 else ""
    print(f"  alpha={alpha:<5} delta={m:+.4f} (std={s:.4f}) p={p:.4f} {sig}")
    sweep_results.append(('alpha', alpha, m, s, p))

print("\n--- noise_std ---")
for noise_std in [0.04, 0.08, 0.15, 0.25]:
    m, s, p = run_paired_delta_v2(30, 100, 0.30, 0.5, noise_std, SEEDS_5)
    sig = "*" if p < 0.05 else ""
    print(f"  noise_std={noise_std:<5} delta={m:+.4f} (std={s:.4f}) p={p:.4f} {sig}")
    sweep_results.append(('noise_std', noise_std, m, s, p))

print("\n--- n_users ---")
for n_users in [10, 30, 50, 100]:
    m, s, p = run_paired_delta_v2(n_users, 100, 0.30, 0.5, 0.08, SEEDS_5)
    sig = "*" if p < 0.05 else ""
    print(f"  n_users={n_users:<5} delta={m:+.4f} (std={s:.4f}) p={p:.4f} {sig}")
    sweep_results.append(('n_users', n_users, m, s, p))

print("\n--- offset_std ---")
for offset_std in [0.0, 0.15, 0.30, 0.50, 0.80]:
    m, s, p = run_paired_delta_v2(30, 100, offset_std, 0.5, 0.08, SEEDS_5)
    sig = "*" if p < 0.05 else ""
    print(f"  offset_std={offset_std:<5} delta={m:+.4f} (std={s:.4f}) p={p:.4f} {sig}")
    sweep_results.append(('offset_std', offset_std, m, s, p))

print("\n--- n_sessions ---")
for n_sessions in [50, 100, 200, 400]:
    m, s, p = run_paired_delta_v2(30, n_sessions, 0.30, 0.5, 0.08, SEEDS_5)
    sig = "*" if p < 0.05 else ""
    print(f"  n_sessions={n_sessions:<5} delta={m:+.4f} (std={s:.4f}) p={p:.4f} {sig}")
    sweep_results.append(('n_sessions', n_sessions, m, s, p))

with open('sweep_v2_results.json', 'w') as f:
    json.dump(sweep_results, f, indent=2)
print("\nSaved sweep_v2_results.json")

=== Sensitivity sweep под reward_v2 ===
baseline: alpha=0.5, noise=0.08, offset_std=0.30, users=30, sessions=100

--- alpha ---
  alpha=0.1   delta=+0.1677 (std=0.0158) p=0.0000 *
  alpha=0.3   delta=+0.1857 (std=0.0284) p=0.0002 *
  alpha=0.5   delta=+0.1844 (std=0.0306) p=0.0003 *
  alpha=0.8   delta=+0.1654 (std=0.0286) p=0.0003 *
  alpha=1.2   delta=+0.1421 (std=0.0319) p=0.0009 *

--- noise_std ---
  noise_std=0.04  delta=+0.1892 (std=0.0221) p=0.0001 *
  noise_std=0.08  delta=+0.1844 (std=0.0306) p=0.0003 *
  noise_std=0.15  delta=+0.1708 (std=0.0152) p=0.0000 *
  noise_std=0.25  delta=+0.1371 (std=0.0172) p=0.0001 *

--- n_users ---
  n_users=10    delta=+0.1965 (std=0.0607) p=0.0029 *
  n_users=30    delta=+0.1844 (std=0.0306) p=0.0003 *
  n_users=50    delta=+0.1721 (std=0.0194) p=0.0001 *
  n_users=100   delta=+0.1717 (std=0.0118) p=0.0000 *

--- offset_std ---
  offset_std=0.0   delta=-0.1184 (std=0.0114) p=0.0000 *
  offset_std=0.15  delta=+0.0445 (std=0.0167) p=0.0060 *
  

adding rule-base exp

In [17]:
def rule_based_action(emotion, valid_arms, fixed_index=0):
    """Детерминированный выбор без знания reward: всегда N-я активность в списке.
    Не смотрит на REWARD_TABLE, не адаптируется, не эксплорирует, не персонализирует."""
    return valid_arms[fixed_index]

def optimal_action_exp3(emotion, valid_arms):
    """Oracle без персонализации (offset_std=0 в Experiment 3) -- эквивалентен argmax по REWARD_TABLE."""
    vals = [REWARD_TABLE[emotion][a] for a in valid_arms]
    return valid_arms[int(np.argmax(vals))]

# Sanity-тест: каждый вариант детерминирован
print("Проверка rule-based на детерминизм (index 0/1/2):")
for idx in [0, 1, 2]:
    print(f"\n  fixed_index={idx}:")
    for emotion in EMOTIONS:
        choices = set(rule_based_action(emotion, EMOTION_GAMES[emotion], idx) for _ in range(50))
        assert len(choices) == 1
        arm = list(choices)[0]
        val = REWARD_TABLE[emotion][arm]
        best_val = max(REWARD_TABLE[emotion][a] for a in EMOTION_GAMES[emotion])
        tag = "= OPTIMAL" if val == best_val else f"suboptimal (best={best_val})"
        print(f"    {emotion}: arm {arm} (value={val}) {tag}")

Проверка rule-based на детерминизм (index 0/1/2):

  fixed_index=0:
    anxiety: arm 0 (value=0.9) = OPTIMAL
    burnout: arm 3 (value=0.9) = OPTIMAL
    sadness: arm 12 (value=0.6) suboptimal (best=0.85)
    overwhelm: arm 9 (value=0.85) = OPTIMAL
    neutral: arm 6 (value=0.85) = OPTIMAL

  fixed_index=1:
    anxiety: arm 1 (value=0.75) suboptimal (best=0.9)
    burnout: arm 4 (value=0.75) suboptimal (best=0.9)
    sadness: arm 13 (value=0.75) suboptimal (best=0.85)
    overwhelm: arm 10 (value=0.8) suboptimal (best=0.85)
    neutral: arm 7 (value=0.8) suboptimal (best=0.85)

  fixed_index=2:
    anxiety: arm 2 (value=0.7) suboptimal (best=0.9)
    burnout: arm 5 (value=0.8) suboptimal (best=0.9)
    sadness: arm 14 (value=0.85) = OPTIMAL
    overwhelm: arm 11 (value=0.75) suboptimal (best=0.85)
    neutral: arm 8 (value=0.75) suboptimal (best=0.85)


In [18]:
def run_seed_experiment3(seed, n_users, n_sessions, fixed_index, alpha=0.5):
    master_rng = np.random.RandomState(seed)
    linucb = LinUCBBandit(alpha=alpha)  # единый бандит, без персонализации

    rewards = {'random': [], 'rule_based': [], 'linucb': [], 'optimal': []}
    for uid in range(n_users):
        for s in range(n_sessions):
            emotion = master_rng.choice(EMOTIONS)
            low, high = STRESS_RANGES[emotion]
            stress = master_rng.uniform(low, high)
            x = make_vector(emotion, stress)
            valid = EMOTION_GAMES[emotion]

            noise = {arm: dict(z=master_rng.normal(), u=master_rng.uniform()) for arm in valid}

            def rf(arm):
                n = noise[arm]
                base = REWARD_TABLE[emotion][arm]
                local_rng = np.random.RandomState(hash((seed, uid, s, arm)) % (2**31))
                r, *_ = reward_v2(base, n['z'], n['u'], local_rng)
                return r

            a = master_rng.choice(valid); rewards['random'].append(rf(a))
            a = rule_based_action(emotion, valid, fixed_index); rewards['rule_based'].append(rf(a))
            a = linucb.select(x, valid); r = rf(a); linucb.update(a, x, r); rewards['linucb'].append(r)
            a = optimal_action_exp3(emotion, valid); rewards['optimal'].append(rf(a))

    return {k: float(np.mean(v)) for k, v in rewards.items()}


SEEDS_5 = [42, 123, 456, 789, 2024]
N_USERS, N_SESSIONS = 30, 100

all_variants = {}
for fixed_index in [0, 1, 2]:
    print(f"\n{'='*60}\nRULE-BASED = index {fixed_index}\n{'='*60}")
    results = {k: [] for k in ['random', 'rule_based', 'linucb', 'optimal']}
    for sd in SEEDS_5:
        r = run_seed_experiment3(sd, N_USERS, N_SESSIONS, fixed_index)
        for k in results:
            results[k].append(r[k])

    for k, v in results.items():
        print(f"  {k:12s} mean={np.mean(v):.4f} std={np.std(v):.4f}")

    t_lr, p_lr = stats.ttest_rel(results['linucb'], results['rule_based'])
    t_lo, p_lo = stats.ttest_rel(results['optimal'], results['linucb'])
    violations = sum(1 for o, l in zip(results['optimal'], results['linucb']) if o < l)

    print(f"\n  LinUCB vs Rule-based (paired): t={t_lr:.3f} p={p_lr:.4f}")
    print(f"  Optimal vs LinUCB (paired):    t={t_lo:.3f} p={p_lo:.4f}")
    print(f"  Optimal violations: {violations}/5")

    all_variants[fixed_index] = results

print("\n\nDone. Compare which fixed_index gives the most informative/defensible baseline.")


RULE-BASED = index 0
  random       mean=0.2267 std=0.0047
  rule_based   mean=0.3144 std=0.0109
  linucb       mean=0.3360 std=0.0498
  optimal      mean=0.4226 std=0.0115

  LinUCB vs Rule-based (paired): t=0.906 p=0.4162
  Optimal vs LinUCB (paired):    t=3.448 p=0.0261
  Optimal violations: 0/5

RULE-BASED = index 1
  random       mean=0.2267 std=0.0047
  rule_based   mean=0.1939 std=0.0093
  linucb       mean=0.3360 std=0.0498
  optimal      mean=0.4226 std=0.0115

  LinUCB vs Rule-based (paired): t=6.685 p=0.0026
  Optimal vs LinUCB (paired):    t=3.448 p=0.0261
  Optimal violations: 0/5

RULE-BASED = index 2
  random       mean=0.2267 std=0.0047
  rule_based   mean=0.1853 std=0.0079
  linucb       mean=0.3360 std=0.0498
  optimal      mean=0.4226 std=0.0115

  LinUCB vs Rule-based (paired): t=5.320 p=0.0060
  Optimal vs LinUCB (paired):    t=3.448 p=0.0261
  Optimal violations: 0/5


Done. Compare which fixed_index gives the most informative/defensible baseline.


add one end-to-end error propagation test


In [21]:
EMOTION_TO_IDX = {'anxiety': 0, 'burnout': 1, 'sadness': 2, 'overwhelm': 3, 'neutral': 4}
IDX_TO_EMOTION = {v: k for k, v in EMOTION_TO_IDX.items()}

CM_MENTAL = np.array([
    [77.7, 7.4, 1.6, 4.1, 9.2],
    [4.5, 63.4, 4.0, 13.5, 14.7],
    [5.4, 6.9, 78.1, 3.5, 6.1],
    [3.1, 17.4, 1.7, 64.6, 13.3],
    [7.4, 9.0, 2.4, 7.7, 73.5],
])
CM_MENTAL = CM_MENTAL / CM_MENTAL.sum(axis=1, keepdims=True)


def perceive_emotion(true_emotion, mode, rng):
    """rng -- обычный np.random.RandomState (не Generator!), для полной
    воспроизводимости между разными версиями numpy / машинами."""
    if mode == 'ground_truth':
        return true_emotion
    elif mode == 'noisy':
        idx = int(rng.uniform(0, 5))
        return EMOTIONS[min(idx, 4)]
    elif mode == 'predicted':
        true_idx = EMOTION_TO_IDX[true_emotion]
        probs = CM_MENTAL[true_idx]
        cdf = np.cumsum(probs)
        u = rng.uniform()
        perceived_idx = int(np.searchsorted(cdf, u))
        perceived_idx = min(perceived_idx, 4)
        return IDX_TO_EMOTION[perceived_idx]
    else:
        raise ValueError(mode)


def optimal_action_ep(emotion, valid_arms):
    vals = [REWARD_TABLE[emotion][a] for a in valid_arms]
    return valid_arms[int(np.argmax(vals))]


print("Проверка: строки CM_MENTAL суммируются в 1.0")
assert np.allclose(CM_MENTAL.sum(axis=1), 1.0)
print("  [OK]")

print("\nПроверка perceive_emotion (predicted соответствует confusion matrix):")
rng = np.random.RandomState(1)
counts = {e: 0 for e in EMOTIONS}
N = 100000
for _ in range(N):
    counts[perceive_emotion('anxiety', 'predicted', rng)] += 1
for e in EMOTIONS:
    empirical = counts[e] / N
    expected = CM_MENTAL[EMOTION_TO_IDX['anxiety']][EMOTION_TO_IDX[e]]
    print(f"  {e}: empirical={empirical:.4f} expected={expected:.4f}")

print("\nПроверка полного детерминизма (одинаковый seed -> одинаковая последовательность):")
rng1 = np.random.RandomState(42)
seq1 = [perceive_emotion('burnout', 'predicted', rng1) for _ in range(10)]
rng2 = np.random.RandomState(42)
seq2 = [perceive_emotion('burnout', 'predicted', rng2) for _ in range(10)]
print(f"  seq1 == seq2: {seq1 == seq2}")

Проверка: строки CM_MENTAL суммируются в 1.0
  [OK]

Проверка perceive_emotion (predicted соответствует confusion matrix):
  anxiety: empirical=0.7768 expected=0.7770
  burnout: empirical=0.0737 expected=0.0740
  sadness: empirical=0.0157 expected=0.0160
  overwhelm: empirical=0.0415 expected=0.0410
  neutral: empirical=0.0923 expected=0.0920

Проверка полного детерминизма (одинаковый seed -> одинаковая последовательность):
  seq1 == seq2: True


In [22]:
def run_seed_error_propagation(seed, n_users, n_sessions, mode, alpha=0.5):
    master_rng = np.random.RandomState(seed)
    linucb = LinUCBBandit(alpha=alpha)

    rewards, optimal_rewards, regrets = [], [], []
    for uid in range(n_users):
        for s in range(n_sessions):
            true_emotion = master_rng.choice(EMOTIONS)
            low, high = STRESS_RANGES[true_emotion]
            stress = master_rng.uniform(low, high)

            perceived_emotion = perceive_emotion(true_emotion, mode, master_rng)  # тот же master_rng
            x = make_vector(perceived_emotion, stress)
            valid = EMOTION_GAMES[perceived_emotion]

            true_valid = EMOTION_GAMES[true_emotion]
            opt_arm = optimal_action_ep(true_emotion, true_valid)

            noise = {arm: dict(z=master_rng.normal(), u=master_rng.uniform())
                     for arm in set(valid) | set(true_valid)}

            def rf(arm):
                n = noise[arm]
                base = REWARD_TABLE[true_emotion][arm]
                local_rng = np.random.RandomState(hash((seed, uid, s, arm, mode)) % (2**31))
                r, *_ = reward_v2(base, n['z'], n['u'], local_rng)
                return r

            a = linucb.select(x, valid)
            r = rf(a)
            linucb.update(a, x, r)
            rewards.append(r)

            r_opt = rf(opt_arm)
            optimal_rewards.append(r_opt)
            regrets.append(r_opt - r)

    return {
        'mean_reward': float(np.mean(rewards)),
        'mean_optimal': float(np.mean(optimal_rewards)),
        'mean_regret': float(np.mean(regrets)),
    }


SEEDS_5 = [42, 123, 456, 789, 2024]
N_USERS, N_SESSIONS = 30, 100
MODES = ['ground_truth', 'predicted', 'noisy']

all_results = {mode: {'reward': [], 'optimal': [], 'regret': []} for mode in MODES}
for mode in MODES:
    print(f"\n--- mode = {mode} ---")
    for sd in SEEDS_5:
        r = run_seed_error_propagation(sd, N_USERS, N_SESSIONS, mode)
        all_results[mode]['reward'].append(r['mean_reward'])
        all_results[mode]['optimal'].append(r['mean_optimal'])
        all_results[mode]['regret'].append(r['mean_regret'])
    rw = all_results[mode]['reward']
    rg = all_results[mode]['regret']
    print(f"  reward: mean={np.mean(rw):.4f} std={np.std(rw):.4f}")
    print(f"  regret: mean={np.mean(rg):.4f} std={np.std(rg):.4f}")

print("\n--- Paired comparisons (vs ground_truth) ---")
for mode in ['predicted', 'noisy']:
    t, p = stats.ttest_rel(all_results['ground_truth']['reward'], all_results[mode]['reward'])
    delta = np.mean(all_results['ground_truth']['reward']) - np.mean(all_results[mode]['reward'])
    pct_loss = delta / np.mean(all_results['ground_truth']['reward']) * 100
    print(f"  ground_truth vs {mode}: delta={delta:+.4f} ({pct_loss:.1f}% loss) t={t:.3f} p={p:.4f}")


--- mode = ground_truth ---
  reward: mean=0.3064 std=0.0436
  regret: mean=0.1109 std=0.0393

--- mode = predicted ---
  reward: mean=0.0804 std=0.0236
  regret: mean=0.3411 std=0.0223

--- mode = noisy ---
  reward: mean=-0.3782 std=0.0178
  regret: mean=0.8004 std=0.0229

--- Paired comparisons (vs ground_truth) ---
  ground_truth vs predicted: delta=+0.2261 (73.8% loss) t=10.049 p=0.0006
  ground_truth vs noisy: delta=+0.6846 (223.4% loss) t=28.664 p=0.0000


rule-based, Optimal, epsilon-greedy, Thompson Sampling

In [6]:
def rule_based_action(emotion, valid_arms):
    return valid_arms[0]

def optimal_action_exp3(emotion, valid_arms, user_offset=None):
    zero = np.zeros(N_ARMS) if user_offset is None else user_offset
    vals = [REWARD_TABLE[emotion][a] + zero[a] for a in valid_arms]
    return valid_arms[int(np.argmax(vals))]


class EpsilonGreedyBandit:
    def __init__(self, epsilon=0.1, n_arms=N_ARMS):
        self.epsilon = epsilon
        self.counts = np.zeros(n_arms)
        self.values = np.zeros(n_arms)

    def select(self, valid_arms, rng):
        if rng.random() < self.epsilon:
            return valid_arms[rng.randint(len(valid_arms))]
        vals = [self.values[a] for a in valid_arms]
        return valid_arms[int(np.argmax(vals))]

    def update(self, arm, reward):
        self.counts[arm] += 1
        n = self.counts[arm]
        self.values[arm] += (reward - self.values[arm]) / n


class ThompsonSamplingBandit:
    """Gaussian Thompson Sampling (Normal-Normal conjugate) -- для непрерывного reward_v2,
    не Beta-Bernoulli."""
    def __init__(self, n_arms=N_ARMS, prior_mean=0.0, prior_var=1.0, noise_var=0.25):
        self.post_mean = np.full(n_arms, prior_mean)
        self.post_var = np.full(n_arms, prior_var)
        self.noise_var = noise_var

    def select(self, valid_arms, rng):
        samples = {a: rng.normal(self.post_mean[a], np.sqrt(self.post_var[a])) for a in valid_arms}
        return max(samples, key=samples.get)

    def update(self, arm, reward):
        prior_var = self.post_var[arm]
        prior_mean = self.post_mean[arm]
        post_var = 1.0 / (1.0/prior_var + 1.0/self.noise_var)
        post_mean = post_var * (prior_mean/prior_var + reward/self.noise_var)
        self.post_mean[arm] = post_mean
        self.post_var[arm] = post_var


# Sanity-тесты на сходимость к лучшему arm
print("Тест: EpsilonGreedyBandit сходится к лучшему arm")
rng = np.random.RandomState(0)
eg = EpsilonGreedyBandit(epsilon=0.1)
for _ in range(2000):
    emotion = 'anxiety'
    valid = EMOTION_GAMES[emotion]
    arm = eg.select(valid, rng)
    base = REWARD_TABLE[emotion][arm]
    r, *_ = reward_v2(base, rng.normal(), rng.uniform(), rng)
    eg.update(arm, r)
best_learned = EMOTION_GAMES['anxiety'][np.argmax(eg.values[EMOTION_GAMES['anxiety']])]
print(f"  Лучший arm по обучению: {best_learned} (истинный лучший: 0)")
assert best_learned == 0
print("  [OK]")

print("\nТест: ThompsonSamplingBandit тоже сходится")
rng = np.random.RandomState(1)
ts = ThompsonSamplingBandit()
for _ in range(2000):
    emotion = 'anxiety'
    valid = EMOTION_GAMES[emotion]
    arm = ts.select(valid, rng)
    base = REWARD_TABLE[emotion][arm]
    r, *_ = reward_v2(base, rng.normal(), rng.uniform(), rng)
    ts.update(arm, r)
best_learned_ts = EMOTION_GAMES['anxiety'][np.argmax(ts.post_mean[EMOTION_GAMES['anxiety']])]
print(f"  Лучший arm по обучению: {best_learned_ts} (истинный лучший: 0)")
assert best_learned_ts == 0
print("  [OK]")

Тест: EpsilonGreedyBandit сходится к лучшему arm
  Лучший arm по обучению: 0 (истинный лучший: 0)
  [OK]

Тест: ThompsonSamplingBandit тоже сходится
  Лучший arm по обучению: 0 (истинный лучший: 0)
  [OK]


In [7]:
def run_seed_experiment3_extended(seed, n_users, n_sessions, alpha=0.5, epsilon=0.1):
    master_rng = np.random.RandomState(seed)
    linucb = LinUCBBandit(alpha=alpha)
    eps_greedy = EpsilonGreedyBandit(epsilon=epsilon)
    thompson = ThompsonSamplingBandit()

    rewards = {'random': [], 'rule_based': [], 'linucb': [], 'epsilon_greedy': [], 'thompson': [], 'optimal': []}
    for uid in range(n_users):
        for s in range(n_sessions):
            emotion = master_rng.choice(EMOTIONS)
            low, high = STRESS_RANGES[emotion]
            stress = master_rng.uniform(low, high)
            x = make_vector(emotion, stress)
            valid = EMOTION_GAMES[emotion]

            noise = {arm: dict(z=master_rng.normal(), u=master_rng.uniform()) for arm in valid}

            def rf(arm):
                n = noise[arm]
                base = REWARD_TABLE[emotion][arm]
                local_rng = np.random.RandomState(hash((seed, uid, s, arm)) % (2**31))
                r, *_ = reward_v2(base, n['z'], n['u'], local_rng)
                return r

            a = master_rng.choice(valid); rewards['random'].append(rf(a))
            a = rule_based_action(emotion, valid); rewards['rule_based'].append(rf(a))
            a = linucb.select(x, valid); r = rf(a); linucb.update(a, x, r); rewards['linucb'].append(r)
            a = eps_greedy.select(valid, master_rng); r = rf(a); eps_greedy.update(a, r); rewards['epsilon_greedy'].append(r)
            a = thompson.select(valid, master_rng); r = rf(a); thompson.update(a, r); rewards['thompson'].append(r)
            a = optimal_action_exp3(emotion, valid); rewards['optimal'].append(rf(a))

    return {k: float(np.mean(v)) for k, v in rewards.items()}


SEEDS_5 = [42, 123, 456, 789, 2024]
N_USERS, N_SESSIONS = 30, 100

results = {k: [] for k in ['random', 'rule_based', 'linucb', 'epsilon_greedy', 'thompson', 'optimal']}
for sd in SEEDS_5:
    r = run_seed_experiment3_extended(sd, N_USERS, N_SESSIONS)
    for k in results:
        results[k].append(r[k])

print(f"\n{'Policy':<16}{'Mean':>10}{'Std':>10}")
for k, v in results.items():
    print(f"{k:<16}{np.mean(v):>10.4f}{np.std(v):>10.4f}")

print("\n--- Paired tests (все vs LinUCB) ---")
for k in ['random', 'rule_based', 'epsilon_greedy', 'thompson']:
    t, p = stats.ttest_rel(results['linucb'], results[k])
    print(f"LinUCB vs {k:16s}: delta={np.mean(results['linucb'])-np.mean(results[k]):+.4f}  t={t:.3f}  p={p:.4f}")

print("\n--- Optimal violations ---")
for k in ['linucb', 'epsilon_greedy', 'thompson']:
    violations = sum(1 for o, x in zip(results['optimal'], results[k]) if o < x)
    print(f"Optimal < {k}: {violations}/5")


Policy                Mean       Std
random              0.2300    0.0112
rule_based          0.3196    0.0071
linucb              0.3535    0.0616
epsilon_greedy      0.3846    0.0226
thompson            0.4084    0.0162
optimal             0.4257    0.0145

--- Paired tests (все vs LinUCB) ---
LinUCB vs random          : delta=+0.1235  t=4.051  p=0.0155
LinUCB vs rule_based      : delta=+0.0339  t=1.208  p=0.2937
LinUCB vs epsilon_greedy  : delta=-0.0311  t=-1.398  p=0.2346
LinUCB vs thompson        : delta=-0.0549  t=-2.050  p=0.1097

--- Optimal violations ---
Optimal < linucb: 0/5
Optimal < epsilon_greedy: 0/5
Optimal < thompson: 0/5


learning curve + regret + cold-start point

In [8]:
def run_seed_with_curves(seed, n_users, n_sessions, offset_std, alpha=0.5, pretrain_episodes=5000):
    """ВАЖНО: сохраняем ТУ ЖЕ последовательность вызовов master_rng, что и в уже
    провалидированной 'corrected' конфигурации -- включая random и pop_best,
    даже если их значения не нужны для regret/learning curve. Это гарантирует,
    что мы смотрим на траекторию уже известного результата, а не считаем новый прогон."""
    master_rng = np.random.RandomState(seed)
    user_offsets = make_user_offsets(n_users, offset_std, seed=seed)
    pretrained = pretrain_population_bandit_v2(pretrain_episodes, seed=seed, alpha=alpha)
    non_pers = clone_bandit(pretrained)
    pers = {uid: LinUCBBandit(alpha=alpha) for uid in range(n_users)}  # cold-start

    reward_pers = np.zeros((n_sessions, n_users))
    reward_nonpers = np.zeros((n_sessions, n_users))
    reward_optimal = np.zeros((n_sessions, n_users))

    for uid in range(n_users):
        offset = user_offsets[uid]
        for s in range(n_sessions):
            emotion = master_rng.choice(EMOTIONS)
            low, high = STRESS_RANGES[emotion]
            stress = master_rng.uniform(low, high)
            x = make_vector(emotion, stress)
            valid = EMOTION_GAMES[emotion]

            noise = {arm: dict(z=master_rng.normal(), u=master_rng.uniform()) for arm in valid}

            def rf(arm):
                n = noise[arm]
                base = REWARD_TABLE[emotion][arm] + offset[arm]
                local_rng = np.random.RandomState(hash((seed, uid, s, arm)) % (2**31))
                r, *_ = reward_v2(base, n['z'], n['u'], local_rng)
                return r

            # Сохраняем те же вызовы master_rng, что в исходной валидированной версии
            a = master_rng.choice(valid)   # 'random' -- выбрасываем результат, но обязательно вызываем
            _ = rf(a)

            a = population_best_action(emotion, valid)  # 'pop_best' -- тоже для сохранения траектории
            _ = rf(a)

            a = non_pers.select(x, valid)
            r = rf(a); non_pers.update(a, x, r)
            reward_nonpers[s, uid] = r

            a = pers[uid].select(x, valid)
            r = rf(a); pers[uid].update(a, x, r)
            reward_pers[s, uid] = r

            opt_arm = optimal_action(emotion, valid, offset)
            reward_optimal[s, uid] = rf(opt_arm)

    return reward_pers, reward_nonpers, reward_optimal


# Sanity check -- должно совпасть с уже известным результатом Corrected (n=5)
SEEDS_5 = [42, 123, 456, 789, 2024]
N_USERS, N_SESSIONS, OFFSET_STD = 30, 100, 0.30

rp_test, rn_test, ro_test = run_seed_with_curves(42, N_USERS, N_SESSIONS, OFFSET_STD)
print(f"Sanity check (seed=42): pers={rp_test.mean():.4f} non_pers={rn_test.mean():.4f}")
print("Ожидаем pers≈0.5244, non_pers≈0.3541 (per-seed значения из корректной конфигурации)")

Sanity check (seed=42): pers=0.5244 non_pers=0.3541
Ожидаем pers≈0.5244, non_pers≈0.3541 (per-seed значения из корректной конфигурации)


In [9]:
all_pers, all_nonpers, all_optimal = [], [], []
for sd in SEEDS_5:
    rp, rn, ro = run_seed_with_curves(sd, N_USERS, N_SESSIONS, OFFSET_STD)
    all_pers.append(rp)
    all_nonpers.append(rn)
    all_optimal.append(ro)

all_pers = np.stack(all_pers)       # (n_seeds, n_sessions, n_users)
all_nonpers = np.stack(all_nonpers)
all_optimal = np.stack(all_optimal)

mean_pers_per_session = all_pers.mean(axis=(0, 2))
mean_nonpers_per_session = all_nonpers.mean(axis=(0, 2))
mean_optimal_per_session = all_optimal.mean(axis=(0, 2))

block_points = [10, 20, 50, 100]
print(f"{'Sessions':<12}{'Pers reward':<15}{'NonPers reward':<17}{'Pers regret':<15}{'NonPers regret':<15}")
for bp in block_points:
    pers_cum = all_pers[:, :bp, :].mean()
    nonpers_cum = all_nonpers[:, :bp, :].mean()
    optimal_cum = all_optimal[:, :bp, :].mean()
    print(f"{bp:<12}{pers_cum:<15.4f}{nonpers_cum:<17.4f}{optimal_cum-pers_cum:<15.4f}{optimal_cum-nonpers_cum:<15.4f}")

diff_per_session = mean_pers_per_session - mean_nonpers_per_session
crossover_sessions = np.where(diff_per_session > 0)[0]
if len(crossover_sessions) > 0:
    print(f"\nПервая сессия, где pers > non_pers (сырое сравнение): session {crossover_sessions[0] + 1}")

window = 10
smoothed_pers = np.convolve(mean_pers_per_session, np.ones(window)/window, mode='valid')
smoothed_nonpers = np.convolve(mean_nonpers_per_session, np.ones(window)/window, mode='valid')
smoothed_diff = smoothed_pers - smoothed_nonpers
crossover_smoothed = np.where(smoothed_diff > 0)[0]
if len(crossover_smoothed) > 0:
    print(f"Cold-start point (скользящее среднее, window=10): session ~{crossover_smoothed[0] + window}")

# Сохраняем для графика
np.save('mean_pers_per_session.npy', mean_pers_per_session)
np.save('mean_nonpers_per_session.npy', mean_nonpers_per_session)
np.save('mean_optimal_per_session.npy', mean_optimal_per_session)

Sessions    Pers reward    NonPers reward   Pers regret    NonPers regret 
10          0.3214         0.3338           0.2829         0.2705         
20          0.3928         0.3156           0.2133         0.2906         
50          0.4686         0.3160           0.1334         0.2860         
100         0.5118         0.3293           0.0902         0.2727         

Первая сессия, где pers > non_pers (сырое сравнение): session 3
Cold-start point (скользящее среднее, window=10): session ~11
